SECTION A: DATA CLEANING

DATA VALIDATION & CORRECTION

Explanation

The Rating column should contain values strictly between 1 and 5. However, the dataset includes:

Invalid values: 0, 6
Missing values: Null (NaN)

To handle this:

First, we detect invalid values using conditions such as:
Rating < 1
Rating > 5
Next, we convert invalid values into NaN so they can be treated uniformly.
Finally, we handle missing values by replacing them with the median, which is preferred because it is less affected by outliers.


In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Rating": [5, 4, 0, 6, np.nan, 3, 2]
})

# Step 1: Replace invalid values with NaN
df["Rating"] = df["Rating"].apply(lambda x: x if 1 <= x <= 5 else np.nan)

# Step 2: Fill missing values properly
df["Rating"] = df["Rating"].fillna(df["Rating"].median())

print(df)

   Rating
0     5.0
1     4.0
2     3.5
3     3.5
4     3.5
5     3.0
6     2.0


TEXT PREPROCESSING

Explanation

The Review_Text column contains unwanted elements such as:

HTML tags
Special characters
Emojis

These must be removed to ensure the text is clean and consistent for analysis.

Steps:

Remove HTML tags
Remove special characters and emojis
Convert text to lowercase

Importance:

Improves data quality
Helps in accurate text analysis
Removes unnecessary noise


In [3]:
import re

df = pd.DataFrame({
    "Review_Text": [
        "<p>Great course! 😊</p>",
        "Bad!!! 😡 @#$%",
        "<br>Average content..."
    ]
})

def clean_text(text):
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower()
    return text

df["Cleaned_Text"] = df["Review_Text"].apply(clean_text)

print(df)

              Review_Text     Cleaned_Text
0  <p>Great course! 😊</p>    great course 
1           Bad!!! 😡 @#$%            bad  
2  <br>Average content...  average content


HANDLING RELATIVE DATES
Explanation

The Review_Date column contains values like:

"Yesterday"
"2 days ago"

These are problematic because they are not in standard datetime format, making it difficult to:

Sort data
Perform time-based analysis

Thus, we convert them into proper datetime values.

In [4]:
from datetime import datetime, timedelta
import pandas as pd

df = pd.DataFrame({
    "Review_Date": ["Yesterday", "2 days ago", "March 5th, 2023"]
})

def convert_date(text):
    today = datetime.today()

    if text.lower() == "yesterday":
        return today - timedelta(days=1)

    elif "days ago" in text:
        days = int(text.split()[0])
        return today - timedelta(days=days)

    else:
        return pd.to_datetime(text, errors='coerce')

df["Formatted_Date"] = df["Review_Date"].apply(convert_date)

print(df)

       Review_Date             Formatted_Date
0        Yesterday 2026-03-24 16:37:57.409237
1       2 days ago 2026-03-23 16:37:57.409262
2  March 5th, 2023 2023-03-05 00:00:00.000000


SECTION B: DATA WRANGLING

MULTI-TABLE INTEGRATION

Explanation

We are given three DataFrames:

DF_Students
DF_Courses
DF_Scores

To combine them:

First, merge students with courses using Course_ID
Then, merge the result with scores using Student_ID

In [5]:
import pandas as pd

df_students = pd.DataFrame({
    "Student_ID": [1, 2, 3],
    "Name": ["A", "B", "C"],
    "Course_ID": [101, 102, 101]
})

df_courses = pd.DataFrame({
    "Course_ID": [101, 102],
    "Course_Name": ["Python", "ML"],
    "Instructor": ["John", "Sara"]
})

df_scores = pd.DataFrame({
    "Student_ID": [1, 2, 3],
    "Score": [85, 90, 78]
})

df_merge1 = pd.merge(df_students, df_courses, on="Course_ID", how="inner")
df_final = pd.merge(df_merge1, df_scores, on="Student_ID", how="inner")

print(df_final)

   Student_ID Name  Course_ID Course_Name Instructor  Score
0           1    A        101      Python       John     85
1           2    B        102          ML       Sara     90
2           3    C        101      Python       John     78


GROUPING & AGGREGATION
Explanation

To find the average score per course, we:

Group data using Course_Name
Apply mean() on scores

In [ ]:
avg_score = df_final.groupby("Course_Name")["Score"].mean().reset_index()
print(avg_score)

DATA ALIGNMENT & UPDATE
Explanation

We receive updated scores in a new DataFrame.
We must update existing values without creating duplicates.

This is done using:

Index alignment
update() function

In [6]:
df_scores = pd.DataFrame({
    "Student_ID": [1, 2, 3],
    "Score": [85, 90, 78]
})

df_update = pd.DataFrame({
    "Student_ID": [2],
    "Score": [95]
})

df_scores.set_index("Student_ID", inplace=True)
df_update.set_index("Student_ID", inplace=True)

df_scores.update(df_update)

df_scores.reset_index(inplace=True)

print(df_scores)

   Student_ID  Score
0           1     85
1           2     95
2           3     78
